# Notebook 07: Standard MAML — XuetangX

**Purpose:** First-Order MAML (FOMAML) benchmark for cold-start session-based MOOC recommendation.

**Protocol:**
- K=5 support pairs per episode (inner-loop adaptation)
- Q=10 query pairs per episode (outer-loop evaluation)
- Test on held-out cold-start users (disjoint from train/val)

**Locked hyperparameters (CLAUDE.md):**
```
alpha_base      = 0.01
num_inner_steps = 5
outer_lr        = 0.001
batch_size      = 32
n_iterations    = 3000
seed            = 20260107
```

**Primary metrics:** HR@10, NDCG@10  
**This is the STANDARD MAML BENCHMARK** — NB08/NB10/NB11 compare against these results.

In [1]:
# [CELL 07-00] Bootstrap: repo root + paths + helpers

import os
# Required for deterministic CuBLAS on CUDA >= 10.2
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
import json
import time
import uuid
import math
import copy
import pickle
import hashlib
import random
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Tuple, Optional
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"[CELL 07-00] start={datetime.now().isoformat(timespec='seconds')}")
print("[CELL 07-00] CWD:", Path.cwd().resolve())


def find_repo_root(start: Path) -> Path:
    """Search upward for PROJECT_STATE.md — single source of truth."""
    start = Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / "PROJECT_STATE.md").exists():
            return p
    raise RuntimeError(
        "Could not find PROJECT_STATE.md in current or parent directories. "
        "Open this notebook from inside the repo."
    )


REPO_ROOT = find_repo_root(Path.cwd())

PATHS = {
    "PROJECT_STATE":  REPO_ROOT / "PROJECT_STATE.md",
    "META_REGISTRY":  REPO_ROOT / "meta.json",
    "DATA_RAW":       REPO_ROOT / "data" / "raw",
    "DATA_INTERIM":   REPO_ROOT / "data" / "interim",
    "DATA_PROCESSED": REPO_ROOT / "data" / "processed",
    "NOTEBOOKS":      REPO_ROOT / "notebooks",
    "REPORTS":        REPO_ROOT / "reports",
    "MODELS":         REPO_ROOT / "models",
    "RUNS":           REPO_ROOT / "runs",
}

for p in PATHS.values():
    Path(p).mkdir(parents=True, exist_ok=True) if str(p).endswith(("reports", "models", "runs")) else None
(PATHS["MODELS"] / "baselines").mkdir(parents=True, exist_ok=True)
(PATHS["MODELS"] / "maml").mkdir(parents=True, exist_ok=True)
(PATHS["REPORTS"]).mkdir(parents=True, exist_ok=True)


def cell_start(cid: str, title: str, **kw) -> float:
    t = time.time()
    print(f"\n[{cid}] {title}")
    print(f"[{cid}] start={datetime.now().isoformat(timespec='seconds')}")
    for k, v in kw.items():
        print(f"[{cid}] {k}={v}")
    return t


def cell_end(cid: str, t0: float, **kw) -> None:
    for k, v in kw.items():
        print(f"[{cid}] {k}={v}")
    print(f"[{cid}] elapsed={time.time()-t0:.2f}s  done")


def write_json_atomic(path: Path, obj: Any, indent: int = 2) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f".tmp_{uuid.uuid4().hex}")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding="utf-8")
    tmp.replace(path)


def read_json(path: Path) -> Any:
    path = Path(path)
    if not path.exists():
        raise RuntimeError(f"Missing required JSON file: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


print(f"[CELL 07-00] REPO_ROOT={REPO_ROOT}")
print("[CELL 07-00] done")

[CELL 07-00] start=2026-04-09T13:16:47
[CELL 07-00] CWD: /home/user/jamalla/anonymous-users-mooc-session-meta/notebooks/xuetangx
[CELL 07-00] REPO_ROOT=/home/user/jamalla/anonymous-users-mooc-session-meta
[CELL 07-00] done


In [2]:
# [CELL 07-01] Seed + global constants

t0 = cell_start("CELL 07-01", "Seed everything + global constants")

GLOBAL_SEED = 20260107
NOTEBOOK_NAME = "07_standard_maml_xuetangx"
DATASET = "xuetangx"


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print(f"[CELL 07-01] WARN: use_deterministic_algorithms: {e}")


seed_everything(GLOBAL_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cell_end("CELL 07-01", t0, seed=GLOBAL_SEED, device=DEVICE)


[CELL 07-01] Seed everything + global constants
[CELL 07-01] start=2026-04-09T13:16:47
[CELL 07-01] seed=20260107
[CELL 07-01] device=cuda
[CELL 07-01] elapsed=0.16s  done


In [3]:
# [CELL 07-02] Run config + data paths

t0 = cell_start("CELL 07-02", "Run config + paths")

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ID  = uuid.uuid4().hex

OUT_DIR       = PATHS["REPORTS"] / NOTEBOOK_NAME / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH   = OUT_DIR / "report.json"
CONFIG_PATH   = OUT_DIR / "config.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

# ── Data paths ────────────────────────────────────────────────
EPISODES_DIR = PATHS["DATA_PROCESSED"] / "xuetangx" / "episodes"
PAIRS_DIR    = PATHS["DATA_PROCESSED"] / "xuetangx" / "pairs"
VOCAB_DIR    = PATHS["DATA_PROCESSED"] / "xuetangx" / "vocab"
MODELS_DIR   = PATHS["MODELS"] / "maml"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

K_SUPPORT = 5
Q_QUERY   = 10

EPISODES_TRAIN = EPISODES_DIR / f"episodes_train_K{K_SUPPORT}_Q{Q_QUERY}.parquet"
EPISODES_VAL   = EPISODES_DIR / f"episodes_val_K{K_SUPPORT}_Q{Q_QUERY}.parquet"
EPISODES_TEST  = EPISODES_DIR / f"episodes_test_K{K_SUPPORT}_Q{Q_QUERY}.parquet"
PAIRS_TRAIN    = PAIRS_DIR / "pairs_train.parquet"
PAIRS_VAL      = PAIRS_DIR / "pairs_val.parquet"
PAIRS_TEST     = PAIRS_DIR / "pairs_test.parquet"
VOCAB_PATH     = VOCAB_DIR / "course2id.json"
CHECKPOINT_PATH = MODELS_DIR / f"maml_standard_{DATASET}.pth"
RESULTS_PATH   = OUT_DIR / f"results_{NOTEBOOK_NAME}.json"

# ── Locked hyperparameters (CLAUDE.md — do NOT change) ────────
ALPHA_BASE      = 0.01
NUM_INNER_STEPS = 5
OUTER_LR        = 0.001
BATCH_SIZE      = 32
N_ITERATIONS    = 3000
VAL_EVERY       = 100
MAX_SEQ_LEN     = 50

# ── GRU architecture (same as NB06) ──────────────────────────
GRU_CFG = {
    "embedding_dim": 64,
    "hidden_dim":    128,
    "num_layers":    1,
    "dropout":       0.0,
}

# ── Ablation configs ─────────────────────────────────────────
ABLATION_SUPPORT_SIZES = [1, 3, 5, 10]
ABLATION_STEPS         = [1, 3, 5, 10]

CFG = {
    "notebook":      NOTEBOOK_NAME,
    "dataset":       DATASET,
    "global_seed":   GLOBAL_SEED,
    "device":        DEVICE,
    "K_support":     K_SUPPORT,
    "Q_query":       Q_QUERY,
    "alpha_base":    ALPHA_BASE,
    "num_inner_steps": NUM_INNER_STEPS,
    "outer_lr":      OUTER_LR,
    "batch_size":    BATCH_SIZE,
    "n_iterations":  N_ITERATIONS,
    "val_every":     VAL_EVERY,
    "max_seq_len":   MAX_SEQ_LEN,
    "gru_cfg":       GRU_CFG,
    "ablation_support_sizes": ABLATION_SUPPORT_SIZES,
    "ablation_steps":         ABLATION_STEPS,
    "inputs": {
        "episodes_train": str(EPISODES_TRAIN),
        "episodes_val":   str(EPISODES_VAL),
        "episodes_test":  str(EPISODES_TEST),
        "pairs_train":    str(PAIRS_TRAIN),
        "pairs_val":      str(PAIRS_VAL),
        "pairs_test":     str(PAIRS_TEST),
        "vocab":          str(VOCAB_PATH),
    },
}

write_json_atomic(CONFIG_PATH, CFG)

# ── Init report / manifest ────────────────────────────────────
write_json_atomic(REPORT_PATH, {
    "run_id": RUN_ID, "notebook": NOTEBOOK_NAME, "run_tag": RUN_TAG,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "metrics": {}, "key_findings": [], "sanity_samples": {},
    "data_fingerprints": {}, "notes": [],
})
write_json_atomic(MANIFEST_PATH, {
    "run_id": RUN_ID, "notebook": NOTEBOOK_NAME, "run_tag": RUN_TAG, "artifacts": []
})

# ── Append to meta.json ───────────────────────────────────────
META = PATHS["META_REGISTRY"]
if not META.exists():
    write_json_atomic(META, {"schema_version": 1, "runs": []})
m = read_json(META)
m["runs"].append({"run_id": RUN_ID, "notebook": NOTEBOOK_NAME,
                   "run_tag": RUN_TAG, "out_dir": str(OUT_DIR)})
write_json_atomic(META, m)

print(f"[CELL 07-02] alpha_base={ALPHA_BASE}  inner_steps={NUM_INNER_STEPS}  outer_lr={OUTER_LR}")
print(f"[CELL 07-02] batch_size={BATCH_SIZE}  n_iterations={N_ITERATIONS}  val_every={VAL_EVERY}")
cell_end("CELL 07-02", t0, out_dir=str(OUT_DIR))


[CELL 07-02] Run config + paths
[CELL 07-02] start=2026-04-09T13:16:47
[CELL 07-02] alpha_base=0.01  inner_steps=5  outer_lr=0.001
[CELL 07-02] batch_size=32  n_iterations=3000  val_every=100
[CELL 07-02] out_dir=/home/user/jamalla/anonymous-users-mooc-session-meta/reports/07_standard_maml_xuetangx/20260409_131647
[CELL 07-02] elapsed=0.05s  done


In [4]:
# [CELL 07-03] Load data: vocab, episodes, pairs

t0 = cell_start("CELL 07-03", "Load vocab + episodes + pairs")

# ── Guard: required files must exist ─────────────────────────
for label, path in [
    ("vocab",          VOCAB_PATH),
    ("episodes_train", EPISODES_TRAIN),
    ("episodes_val",   EPISODES_VAL),
    ("episodes_test",  EPISODES_TEST),
    ("pairs_train",    PAIRS_TRAIN),
    ("pairs_val",      PAIRS_VAL),
    ("pairs_test",     PAIRS_TEST),
]:
    if not Path(path).exists():
        raise RuntimeError(
            f"Missing {label}: {path}\n"
            "Run notebooks 01→02→03→03b→04→05 first."
        )

# ── Vocab ────────────────────────────────────────────────────
course2id = read_json(VOCAB_PATH)
id2course  = {int(v): k for k, v in course2id.items()}
n_items    = len(course2id)
print(f"[CELL 07-03] Vocabulary: {n_items} courses")

# ── Episodes ─────────────────────────────────────────────────
episodes_train = pd.read_parquet(EPISODES_TRAIN)
episodes_val   = pd.read_parquet(EPISODES_VAL)
episodes_test  = pd.read_parquet(EPISODES_TEST)

print(f"[CELL 07-03] episodes_train: {len(episodes_train):,} ({episodes_train['user_id'].nunique():,} users)")
print(f"[CELL 07-03] episodes_val:   {len(episodes_val):,} ({episodes_val['user_id'].nunique():,} users)")
print(f"[CELL 07-03] episodes_test:  {len(episodes_test):,} ({episodes_test['user_id'].nunique():,} users)")

# ── Pairs ────────────────────────────────────────────────────
pairs_train = pd.read_parquet(PAIRS_TRAIN)
pairs_val   = pd.read_parquet(PAIRS_VAL)
pairs_test  = pd.read_parquet(PAIRS_TEST)

print(f"[CELL 07-03] pairs_train: {len(pairs_train):,}")
print(f"[CELL 07-03] pairs_val:   {len(pairs_val):,}")
print(f"[CELL 07-03] pairs_test:  {len(pairs_test):,}")

# ── Index pairs by pair_id for fast episode lookup ────────────
pairs_train_idx = pairs_train.set_index("pair_id")
pairs_val_idx   = pairs_val.set_index("pair_id")
pairs_test_idx  = pairs_test.set_index("pair_id")

cell_end("CELL 07-03", t0, n_items=n_items)


[CELL 07-03] Load vocab + episodes + pairs
[CELL 07-03] start=2026-04-09T13:16:47
[CELL 07-03] Vocabulary: 1517 courses
[CELL 07-03] episodes_train: 113,920 (4,645 users)
[CELL 07-03] episodes_val:   942 (942 users)
[CELL 07-03] episodes_test:  986 (986 users)
[CELL 07-03] pairs_train: 344,532
[CELL 07-03] pairs_val:   69,663
[CELL 07-03] pairs_test:  73,501
[CELL 07-03] n_items=1517
[CELL 07-03] elapsed=0.46s  done


In [5]:
# [CELL 07-04] Evaluation metrics: HR@1/5/10, NDCG@10, MRR

t0 = cell_start("CELL 07-04", "Define evaluation metrics")


def ndcg_at_k(ranked_items: np.ndarray, true_item: int, k: int = 10) -> float:
    """NDCG@K = 1/log2(rank+2) if true_item in top-K, else 0."""
    for i, item in enumerate(ranked_items[:k]):
        if item == true_item:
            return 1.0 / math.log2(i + 2)
    return 0.0


def compute_metrics(
    scores: np.ndarray,
    labels: np.ndarray,
) -> Dict[str, float]:
    """
    Compute HR@1, HR@5, HR@10, NDCG@10, MRR.

    Args:
        scores: (n_samples, n_items) float score matrix (higher = more relevant)
        labels: (n_samples,) int true item indices

    Returns:
        dict with keys: hr1, hr5, hr10, ndcg10, mrr
    """
    n = len(labels)
    ranked = np.argsort(-scores, axis=1)  # descending rank

    hr1_vals    = []
    hr5_vals    = []
    hr10_vals   = []
    ndcg10_vals = []
    mrr_vals    = []

    for i in range(n):
        true = int(labels[i])
        rank_arr = ranked[i]

        hr1_vals.append(1.0 if rank_arr[0] == true else 0.0)
        hr5_vals.append(1.0 if true in rank_arr[:5] else 0.0)
        hr10_vals.append(1.0 if true in rank_arr[:10] else 0.0)
        ndcg10_vals.append(ndcg_at_k(rank_arr, true, k=10))

        # MRR: reciprocal rank (unbounded)
        pos = np.where(rank_arr == true)[0]
        mrr_vals.append(1.0 / (pos[0] + 1) if len(pos) > 0 else 0.0)

    return {
        "hr1":    float(np.mean(hr1_vals)),
        "hr5":    float(np.mean(hr5_vals)),
        "hr10":   float(np.mean(hr10_vals)),
        "ndcg10": float(np.mean(ndcg10_vals)),
        "mrr":    float(np.mean(mrr_vals)),
        "n":      n,
    }


cell_end("CELL 07-04", t0)


[CELL 07-04] Define evaluation metrics
[CELL 07-04] start=2026-04-09T13:16:47
[CELL 07-04] elapsed=0.00s  done


In [6]:
# [CELL 07-05] GRURecommender — same architecture as NB06

t0 = cell_start("CELL 07-05", "Define GRURecommender")


class GRURecommender(nn.Module):
    """GRU-based sequential recommender.

    Architecture (matches NB06):
        Embedding(n_items, embedding_dim) → GRU(embedding_dim, hidden_dim) → Linear(hidden_dim, n_items)
    """

    def __init__(
        self,
        n_items: int,
        embedding_dim: int = 64,
        hidden_dim: int = 128,
        num_layers: int = 1,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.n_items      = n_items
        self.embedding_dim = embedding_dim
        self.hidden_dim   = hidden_dim
        self.num_layers   = num_layers

        self.embedding = nn.Embedding(n_items, embedding_dim, padding_idx=0)
        # dropout only applied between layers (requires num_layers > 1)
        gru_dropout = dropout if num_layers > 1 else 0.0
        self.gru = nn.GRU(
            embedding_dim, hidden_dim, num_layers,
            batch_first=True, dropout=gru_dropout
        )
        self.fc = nn.Linear(hidden_dim, n_items)

    def forward(self, seq: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        """
        Args:
            seq:     (batch, max_len) padded item-id sequences
            lengths: (batch,) actual sequence lengths
        Returns:
            logits: (batch, n_items)
        """
        emb = self.embedding(seq)                       # (B, L, E)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h_n = self.gru(packed)                       # h_n: (layers, B, H)
        h_last = h_n[-1]                                # (B, H)
        return self.fc(h_last)                          # (B, n_items)


# Quick sanity check
_m = GRURecommender(n_items=100, **GRU_CFG)
_x = torch.randint(1, 100, (4, 10))
_l = torch.tensor([10, 8, 5, 3])
_o = _m(_x, _l)
assert _o.shape == (4, 100), f"Expected (4,100) got {_o.shape}"
del _m, _x, _l, _o

cell_end("CELL 07-05", t0)


[CELL 07-05] Define GRURecommender
[CELL 07-05] start=2026-04-09T13:16:48
[CELL 07-05] elapsed=0.05s  done


In [7]:
# [CELL 07-06] functional_forward — uses torch.func.functional_call (fast CUDA GRU)

t0 = cell_start("CELL 07-06", "Define functional_forward for FOMAML")

from torch.func import functional_call

def functional_forward(
    model: GRURecommender,
    params: Dict[str, torch.Tensor],
    seqs: torch.Tensor,
    lengths: torch.Tensor,
) -> torch.Tensor:
    """
    Functional forward using torch.func.functional_call.
    Runs the model's native optimized CUDA GRU with the given params dict.
    Vastly faster than manual step-by-step GRU loop.
    """
    return functional_call(model, params, (seqs, lengths))


cell_end("CELL 07-06", t0)


[CELL 07-06] Define functional_forward for FOMAML
[CELL 07-06] start=2026-04-09T13:16:48
[CELL 07-06] elapsed=0.00s  done


In [8]:
# [CELL 07-07] pad_sequences + get_episode_data helpers

t0 = cell_start("CELL 07-07", "Define pad_sequences + get_episode_data")


def pad_sequences(
    sequences: List[List[int]],
    max_len: int,
    pad_value: int = 0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Pad a list of variable-length sequences to max_len.

    Returns:
        padded:  (batch, max_len) LongTensor
        lengths: (batch,)         LongTensor of actual lengths (clipped to max_len)
    """
    batch = len(sequences)
    padded = torch.full((batch, max_len), pad_value, dtype=torch.long)
    lengths = torch.zeros(batch, dtype=torch.long)
    for i, seq in enumerate(sequences):
        seq = seq[:max_len]
        l   = len(seq)
        if l > 0:
            padded[i, :l] = torch.tensor(seq, dtype=torch.long)
        lengths[i] = max(l, 1)   # avoid zero-length (GRU pack requirement)
    return padded, lengths


def get_episode_data(
    episode: pd.Series,
    pairs_idx: pd.DataFrame,
    max_seq_len: int = MAX_SEQ_LEN,
    device: str = "cpu",
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Fetch and tensorise support/query batches for one episode.

    The pairs table must have columns:
        prefix_ids (list[int] or stringified list), label_id (int)

    Returns:
        sup_seqs, sup_lengths, sup_labels  — support batch
        qry_seqs, qry_lengths, qry_labels  — query batch
    """
    sup_ids = episode["support_pair_ids"]
    qry_ids = episode["query_pair_ids"]

    if isinstance(sup_ids, str):
        import ast
        sup_ids = ast.literal_eval(sup_ids)
        qry_ids = ast.literal_eval(qry_ids)

    def fetch_batch(pair_ids):
        rows  = pairs_idx.loc[pair_ids]
        seqs  = []
        for prefix in rows["prefix"]:
            if isinstance(prefix, str):
                import ast
                prefix = ast.literal_eval(prefix)
            seqs.append(list(prefix))
        labels = rows["label"].tolist()
        return seqs, labels

    sup_seqs_list, sup_labels_list = fetch_batch(sup_ids)
    qry_seqs_list, qry_labels_list = fetch_batch(qry_ids)

    sup_seqs, sup_lengths = pad_sequences(sup_seqs_list, max_seq_len)
    qry_seqs, qry_lengths = pad_sequences(qry_seqs_list, max_seq_len)

    sup_labels = torch.tensor(sup_labels_list, dtype=torch.long)
    qry_labels = torch.tensor(qry_labels_list, dtype=torch.long)

    return (
        sup_seqs.to(device),  sup_lengths.to(device),  sup_labels.to(device),
        qry_seqs.to(device),  qry_lengths.to(device),  qry_labels.to(device),
    )


cell_end("CELL 07-07", t0)


[CELL 07-07] Define pad_sequences + get_episode_data
[CELL 07-07] start=2026-04-09T13:16:48
[CELL 07-07] elapsed=0.00s  done


In [9]:
# [CELL 07-08] MAML training loop (3000 iterations, val every 100, save best HR@10 checkpoint)

t0_train = cell_start(
    "CELL 07-08", "Standard FOMAML training",
    n_iterations=N_ITERATIONS, batch_size=BATCH_SIZE,
    alpha_base=ALPHA_BASE, inner_steps=NUM_INNER_STEPS, outer_lr=OUTER_LR
)

seed_everything(GLOBAL_SEED)   # deterministic training run

# ── Initialise meta-model ─────────────────────────────────────
meta_model = GRURecommender(
    n_items=n_items,
    embedding_dim=GRU_CFG["embedding_dim"],
    hidden_dim=GRU_CFG["hidden_dim"],
    num_layers=GRU_CFG["num_layers"],
    dropout=GRU_CFG["dropout"],
).to(DEVICE)

n_params = sum(p.numel() for p in meta_model.parameters())
print(f"[CELL 07-08] Meta-model parameters: {n_params:,}")

meta_optimizer = torch.optim.Adam(meta_model.parameters(), lr=OUTER_LR)

# ── Prepare training index ────────────────────────────────────
train_indices = list(range(len(episodes_train)))

# ── Tracking ─────────────────────────────────────────────────
train_losses  = []
val_hr10_hist = []
best_val_hr10 = -1.0
best_iter     = 0
best_state    = None

# ── Outer loop ───────────────────────────────────────────────
for iteration in range(1, N_ITERATIONS + 1):
    meta_model.train()
    meta_optimizer.zero_grad()

    # Sample batch of task indices
    batch_idx = np.random.choice(train_indices, size=min(BATCH_SIZE, len(train_indices)), replace=False)
    batch_episodes = episodes_train.iloc[batch_idx]

    outer_loss = torch.tensor(0.0, device=DEVICE)
    n_valid    = 0
    _n_errors  = 0  # diagnostic counter

    # ── Debug: print first iteration diagnostics ────────────
    if iteration == 1:
        ep0 = batch_episodes.iloc[0]
        sup_ids = ep0["support_pair_ids"]
        print(f"[DEBUG] iter=1 sup_ids type={type(sup_ids)} val={sup_ids}")
        print(f"[DEBUG] pairs_train_idx index dtype={pairs_train_idx.index.dtype}")
        print(f"[DEBUG] pairs_train_idx columns={list(pairs_train_idx.columns)}")

    for _, episode in batch_episodes.iterrows():
        try:
            sup_seqs, sup_lengths, sup_labels, \
            qry_seqs, qry_lengths, qry_labels = get_episode_data(
                episode, pairs_train_idx, MAX_SEQ_LEN, DEVICE
            )
        except Exception as _e:
            _n_errors += 1
            if _n_errors <= 3:
                print(f"[CELL 07-08] Episode error (iter={iteration}): {type(_e).__name__}: {_e}")
            continue

        # ── Inner loop (FOMAML: create_graph=False) ──────────
        params = {n: p.clone() for n, p in meta_model.named_parameters()}

        for _step in range(NUM_INNER_STEPS):
            logits = functional_forward(meta_model, params, sup_seqs, sup_lengths)
            loss_s = F.cross_entropy(logits, sup_labels)
            grads  = torch.autograd.grad(
                loss_s, list(params.values()), create_graph=False
            )
            params = {
                n: p - ALPHA_BASE * g
                for (n, p), g in zip(params.items(), grads)
            }

        # ── Query loss (outer loss) ───────────────────────────
        qry_logits = functional_forward(meta_model, params, qry_seqs, qry_lengths)
        loss_q     = F.cross_entropy(qry_logits, qry_labels)
        outer_loss = outer_loss + loss_q
        n_valid   += 1

    if n_valid == 0:
        continue

    (outer_loss / n_valid).backward()
    torch.nn.utils.clip_grad_norm_(meta_model.parameters(), max_norm=5.0)
    meta_optimizer.step()

    train_losses.append(float(outer_loss.item() / n_valid))

    if iteration % 10 == 0:
        print(f"[CELL 07-08] iter={iteration}/{N_ITERATIONS}  loss={train_losses[-1]:.4f}")

    # ── Validation ───────────────────────────────────────────
    if iteration % VAL_EVERY == 0:
        meta_model.train()  # keep train mode — inner loop needs cuDNN backward
        val_scores_all = []
        val_labels_all = []

        for _, ep in episodes_val.iterrows():
                try:
                    sup_s, sup_l, sup_lb, qry_s, qry_l, qry_lb = get_episode_data(
                        ep, pairs_val_idx, MAX_SEQ_LEN, DEVICE
                    )
                except (KeyError, ValueError):
                    continue

                # Adapt on support — params_v must require grad, so clone OUTSIDE no_grad
                params_v = {n: p.clone() for n, p in meta_model.named_parameters()}
                for _step in range(NUM_INNER_STEPS):
                    lg_v = functional_forward(meta_model, params_v, sup_s, sup_l)
                    ls_v = F.cross_entropy(lg_v, sup_lb)
                    gv   = torch.autograd.grad(ls_v, list(params_v.values()), create_graph=False)
                    params_v = {n: p - ALPHA_BASE * g for (n, p), g in zip(params_v.items(), gv)}

                with torch.no_grad():
                    qry_lg = functional_forward(meta_model, params_v, qry_s, qry_l)
                val_scores_all.append(qry_lg.detach().cpu().numpy())
                val_labels_all.extend(qry_lb.cpu().tolist())

        if len(val_labels_all) > 0:
            val_scores_np = np.vstack(val_scores_all)
            val_labels_np = np.array(val_labels_all)
            val_m = compute_metrics(val_scores_np, val_labels_np)
            val_hr10_hist.append((iteration, val_m["hr10"]))

            if val_m["hr10"] > best_val_hr10:
                best_val_hr10 = val_m["hr10"]
                best_iter     = iteration
                best_state    = copy.deepcopy(meta_model.state_dict())
                torch.save(best_state, CHECKPOINT_PATH)

            print(
                f"[CELL 07-08] iter={iteration:4d} "
                f"train_loss={train_losses[-1]:.4f}  "
                f"val_HR@10={val_m['hr10']*100:.2f}%  "
                f"best_HR@10={best_val_hr10*100:.2f}%@{best_iter}"
            )

print(f"\n[CELL 07-08] Training complete.")
print(f"[CELL 07-08] Best val HR@10: {best_val_hr10*100:.2f}% at iteration {best_iter}")
print(f"[CELL 07-08] Checkpoint saved: {CHECKPOINT_PATH}")

cell_end("CELL 07-08", t0_train, best_iter=best_iter, best_val_hr10=f"{best_val_hr10*100:.2f}%")


[CELL 07-08] Standard FOMAML training
[CELL 07-08] start=2026-04-09T13:16:48
[CELL 07-08] n_iterations=3000
[CELL 07-08] batch_size=32
[CELL 07-08] alpha_base=0.01
[CELL 07-08] inner_steps=5
[CELL 07-08] outer_lr=0.001
[CELL 07-08] Meta-model parameters: 367,277
[DEBUG] iter=1 sup_ids type=<class 'numpy.ndarray'> val=[166757 166758 166759 166760 166761]
[DEBUG] pairs_train_idx index dtype=int64
[DEBUG] pairs_train_idx columns=['user_id', 'session_id', 'prefix', 'label', 'label_ts_epoch', 'prefix_len']
[CELL 07-08] iter=10/3000  loss=7.2626
[CELL 07-08] iter=20/3000  loss=7.1187
[CELL 07-08] iter=30/3000  loss=6.7883
[CELL 07-08] iter=40/3000  loss=6.0225
[CELL 07-08] iter=50/3000  loss=6.4464
[CELL 07-08] iter=60/3000  loss=5.9484
[CELL 07-08] iter=70/3000  loss=6.0266
[CELL 07-08] iter=80/3000  loss=6.3174
[CELL 07-08] iter=90/3000  loss=6.0859
[CELL 07-08] iter=100/3000  loss=5.8408
[CELL 07-08] iter= 100 train_loss=5.8408  val_HR@10=25.12%  best_HR@10=25.12%@100
[CELL 07-08] iter=1

In [14]:
# [CELL 07-09] Test evaluation — load best checkpoint, evaluate on test episodes

t0 = cell_start("CELL 07-09", "Test evaluation (best checkpoint)")

# Load best checkpoint
if not CHECKPOINT_PATH.exists():
    raise RuntimeError(f"Checkpoint missing: {CHECKPOINT_PATH}. Run training cell first.")

meta_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
meta_model.train()  # keep train mode — inner loop needs cuDNN backward
print(f"[CELL 07-09] Loaded checkpoint from iter {best_iter}: {CHECKPOINT_PATH}")

test_scores_all = []
test_labels_all = []
n_skipped = 0

for _, ep in episodes_test.iterrows():
    try:
        sup_s, sup_l, sup_lb, qry_s, qry_l, qry_lb = get_episode_data(
            ep, pairs_test_idx, MAX_SEQ_LEN, DEVICE
        )
    except (KeyError, ValueError):
        n_skipped += 1
        continue

    # Adapt on support set (few-shot, K=5)
    params_t = {n: p.clone() for n, p in meta_model.named_parameters()}
    for _step in range(NUM_INNER_STEPS):
        lg_t = functional_forward(meta_model, params_t, sup_s, sup_l)
        ls_t = F.cross_entropy(lg_t, sup_lb)
        gt   = torch.autograd.grad(ls_t, list(params_t.values()), create_graph=False)
        params_t = {n: p - ALPHA_BASE * g for (n, p), g in zip(params_t.items(), gt)}

    with torch.no_grad():
        qry_lg = functional_forward(meta_model, params_t, qry_s, qry_l)
    test_scores_all.append(qry_lg.detach().cpu().numpy())
    test_labels_all.extend(qry_lb.cpu().tolist())

print(f"[CELL 07-09] episodes evaluated: {len(test_scores_all)}, skipped: {n_skipped}")

test_scores_np = np.vstack(test_scores_all)
test_labels_np = np.array(test_labels_all)

test_metrics = compute_metrics(test_scores_np, test_labels_np)

test_hr1    = test_metrics["hr1"]
test_hr5    = test_metrics["hr5"]
test_hr10   = test_metrics["hr10"]
test_ndcg10 = test_metrics["ndcg10"]
test_mrr    = test_metrics["mrr"]

print(f"\n[CELL 07-09] TEST RESULTS (K={K_SUPPORT} support, Q={Q_QUERY} query):")
print(f"  HR@1    = {test_hr1*100:.2f}%")
print(f"  HR@5    = {test_hr5*100:.2f}%")
print(f"  HR@10   = {test_hr10*100:.2f}%   <- PRIMARY")
print(f"  NDCG@10 = {test_ndcg10*100:.2f}%   <- PRIMARY")
print(f"  MRR     = {test_mrr*100:.2f}%")

cell_end("CELL 07-09", t0, n_test_samples=len(test_labels_all))


[CELL 07-09] Test evaluation (best checkpoint)
[CELL 07-09] start=2026-04-09T14:05:08
[CELL 07-09] Loaded checkpoint from iter 2800: /home/user/jamalla/anonymous-users-mooc-session-meta/models/maml/maml_standard_xuetangx.pth
[CELL 07-09] episodes evaluated: 986, skipped: 0

[CELL 07-09] TEST RESULTS (K=5 support, Q=10 query):
  HR@1    = 23.36%
  HR@5    = 39.54%
  HR@10   = 47.46%   <- PRIMARY
  NDCG@10 = 34.35%   <- PRIMARY
  MRR     = 31.49%
[CELL 07-09] n_test_samples=9860
[CELL 07-09] elapsed=18.74s  done


In [16]:
# [CELL 07-10] Ablation: support set size K ∈ {1, 3, 5, 10}

t0 = cell_start("CELL 07-10", "Ablation: varying support set size K")

meta_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
meta_model.train()  # keep train mode — inner loop needs cuDNN backward

ablation_support_results = {}

for K_test in ABLATION_SUPPORT_SIZES:
    scores_list = []
    labels_list = []

    for _, ep in episodes_test.iterrows():
        try:
            sup_s, sup_l, sup_lb, qry_s, qry_l, qry_lb = get_episode_data(
                ep, pairs_test_idx, MAX_SEQ_LEN, DEVICE
            )
        except (KeyError, ValueError):
            continue

        # Truncate support to K_test
        k_use = min(K_test, sup_s.shape[0])
        if k_use == 0:
            continue
        s_s  = sup_s[:k_use]
        s_l  = sup_l[:k_use]
        s_lb = sup_lb[:k_use]

        params_k = {n: p.clone() for n, p in meta_model.named_parameters()}
        for _step in range(NUM_INNER_STEPS):
            with torch.enable_grad():
                lg_k = functional_forward(meta_model, params_k, s_s, s_l)
                ls_k = F.cross_entropy(lg_k, s_lb)
                gk   = torch.autograd.grad(ls_k, list(params_k.values()), create_graph=False)
            params_k = {n: p - ALPHA_BASE * g for (n, p), g in zip(params_k.items(), gk)}

        with torch.no_grad():
            lg_q = functional_forward(meta_model, params_k, qry_s, qry_l)
        scores_list.append(lg_q.detach().cpu().numpy())
        labels_list.extend(qry_lb.cpu().tolist())

    if len(labels_list) == 0:
        print(f"[CELL 07-10] K={K_test}: no valid episodes")
        continue

    m = compute_metrics(np.vstack(scores_list), np.array(labels_list))
    ablation_support_results[K_test] = m
    print(
        f"[CELL 07-10] K={K_test:2d} | "
        f"HR@1={m['hr1']*100:.2f}%  HR@5={m['hr5']*100:.2f}%  "
        f"HR@10={m['hr10']*100:.2f}%  NDCG@10={m['ndcg10']*100:.2f}%  "
        f"MRR={m['mrr']*100:.2f}%  (n={m['n']})"
    )

print("\n[CELL 07-10] Ablation (support size) summary:")
print(f"{'K':>4}  {'HR@1':>7}  {'HR@5':>7}  {'HR@10':>7}  {'NDCG@10':>9}  {'MRR':>7}")
print("-" * 50)
for K_test in ABLATION_SUPPORT_SIZES:
    if K_test in ablation_support_results:
        m = ablation_support_results[K_test]
        print(f"{K_test:>4}  {m['hr1']*100:>6.2f}%  {m['hr5']*100:>6.2f}%  "
              f"{m['hr10']*100:>6.2f}%  {m['ndcg10']*100:>8.2f}%  {m['mrr']*100:>6.2f}%")

cell_end("CELL 07-10", t0)


[CELL 07-10] Ablation: varying support set size K
[CELL 07-10] start=2026-04-09T14:07:55
[CELL 07-10] K= 1 | HR@1=21.06%  HR@5=38.26%  HR@10=46.03%  NDCG@10=32.62%  MRR=29.66%  (n=9860)
[CELL 07-10] K= 3 | HR@1=22.55%  HR@5=39.31%  HR@10=46.91%  NDCG@10=33.76%  MRR=30.89%  (n=9860)
[CELL 07-10] K= 5 | HR@1=23.36%  HR@5=39.54%  HR@10=47.46%  NDCG@10=34.35%  MRR=31.49%  (n=9860)
[CELL 07-10] K=10 | HR@1=23.36%  HR@5=39.54%  HR@10=47.46%  NDCG@10=34.35%  MRR=31.49%  (n=9860)

[CELL 07-10] Ablation (support size) summary:
   K     HR@1     HR@5    HR@10    NDCG@10      MRR
--------------------------------------------------
   1   21.06%   38.26%   46.03%     32.62%   29.66%
   3   22.55%   39.31%   46.91%     33.76%   30.89%
   5   23.36%   39.54%   47.46%     34.35%   31.49%
  10   23.36%   39.54%   47.46%     34.35%   31.49%
[CELL 07-10] elapsed=73.15s  done


In [18]:
# [CELL 07-11] Ablation: adaptation steps ∈ {1, 3, 5, 10}

t0 = cell_start("CELL 07-11", "Ablation: varying adaptation steps")

meta_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
meta_model.train()  # keep train mode — inner loop needs cuDNN backward

ablation_steps_results = {}

for n_steps in ABLATION_STEPS:
    scores_list = []
    labels_list = []

    for _, ep in episodes_test.iterrows():
        try:
            sup_s, sup_l, sup_lb, qry_s, qry_l, qry_lb = get_episode_data(
                ep, pairs_test_idx, MAX_SEQ_LEN, DEVICE
            )
        except (KeyError, ValueError):
            continue

        params_s = {n: p.clone() for n, p in meta_model.named_parameters()}
        for _step in range(n_steps):
            with torch.enable_grad():
                lg_s = functional_forward(meta_model, params_s, sup_s, sup_l)
                ls_s = F.cross_entropy(lg_s, sup_lb)
                gs   = torch.autograd.grad(ls_s, list(params_s.values()), create_graph=False)
            params_s = {n: p - ALPHA_BASE * g for (n, p), g in zip(params_s.items(), gs)}

        with torch.no_grad():
            lg_q = functional_forward(meta_model, params_s, qry_s, qry_l)
        scores_list.append(lg_q.detach().cpu().numpy())
        labels_list.extend(qry_lb.cpu().tolist())

    if len(labels_list) == 0:
        print(f"[CELL 07-11] steps={n_steps}: no valid episodes")
        continue

    m = compute_metrics(np.vstack(scores_list), np.array(labels_list))
    ablation_steps_results[n_steps] = m
    print(
        f"[CELL 07-11] steps={n_steps:2d} | "
        f"HR@1={m['hr1']*100:.2f}%  HR@5={m['hr5']*100:.2f}%  "
        f"HR@10={m['hr10']*100:.2f}%  NDCG@10={m['ndcg10']*100:.2f}%  "
        f"MRR={m['mrr']*100:.2f}%  (n={m['n']})"
    )

print("\n[CELL 07-11] Ablation (adaptation steps) summary:")
print(f"{'Steps':>6}  {'HR@1':>7}  {'HR@5':>7}  {'HR@10':>7}  {'NDCG@10':>9}  {'MRR':>7}")
print("-" * 55)
for n_steps in ABLATION_STEPS:
    if n_steps in ablation_steps_results:
        m = ablation_steps_results[n_steps]
        print(f"{n_steps:>6}  {m['hr1']*100:>6.2f}%  {m['hr5']*100:>6.2f}%  "
              f"{m['hr10']*100:>6.2f}%  {m['ndcg10']*100:>8.2f}%  {m['mrr']*100:>6.2f}%")

cell_end("CELL 07-11", t0)


[CELL 07-11] Ablation: varying adaptation steps
[CELL 07-11] start=2026-04-09T14:11:06
[CELL 07-11] steps= 1 | HR@1=21.31%  HR@5=37.46%  HR@10=45.67%  NDCG@10=32.46%  MRR=29.58%  (n=9860)
[CELL 07-11] steps= 3 | HR@1=22.51%  HR@5=38.79%  HR@10=46.55%  NDCG@10=33.53%  MRR=30.71%  (n=9860)
[CELL 07-11] steps= 5 | HR@1=23.36%  HR@5=39.54%  HR@10=47.46%  NDCG@10=34.35%  MRR=31.49%  (n=9860)
[CELL 07-11] steps=10 | HR@1=24.12%  HR@5=40.88%  HR@10=48.68%  NDCG@10=35.41%  MRR=32.47%  (n=9860)

[CELL 07-11] Ablation (adaptation steps) summary:
 Steps     HR@1     HR@5    HR@10    NDCG@10      MRR
-------------------------------------------------------
     1   21.31%   37.46%   45.67%     32.46%   29.58%
     3   22.51%   38.79%   46.55%     33.53%   30.71%
     5   23.36%   39.54%   47.46%     34.35%   31.49%
    10   24.12%   40.88%   48.68%     35.41%   32.47%
[CELL 07-11] elapsed=71.02s  done


In [19]:
# [CELL 07-12] Standard result block (CLAUDE.md mandatory format)

print("=" * 55)
print(f"RESULTS — {NOTEBOOK_NAME} — {DATASET}")
print("=" * 55)
print(f"Protocol:      K={K_SUPPORT} support, Q={Q_QUERY} query")
print(f"Test episodes: {len(episodes_test)}")
print(f"Seed:          {GLOBAL_SEED}")
print()
print(f"HR@1:          {test_hr1*100:.2f}%")
print(f"HR@5:          {test_hr5*100:.2f}%")
print(f"HR@10:         {test_hr10*100:.2f}%   <- PRIMARY")
print(f"NDCG@10:       {test_ndcg10*100:.2f}%   <- PRIMARY")
print(f"MRR:           {test_mrr*100:.2f}%")
print()
print(f"Best val iter: {best_iter}")
print(f"Best val HR@10:{best_val_hr10*100:.2f}%")
print("=" * 55)

RESULTS — 07_standard_maml_xuetangx — xuetangx
Protocol:      K=5 support, Q=10 query
Test episodes: 986
Seed:          20260107

HR@1:          23.36%
HR@5:          39.54%
HR@10:         47.46%   <- PRIMARY
NDCG@10:       34.35%   <- PRIMARY
MRR:           31.49%

Best val iter: 2800
Best val HR@10:49.12%


In [20]:
# [CELL 07-13] Save results JSON

t0 = cell_start("CELL 07-13", "Save results JSON")

results = {
    "run_id":        RUN_ID,
    "notebook":      NOTEBOOK_NAME,
    "dataset":       DATASET,
    "run_tag":       RUN_TAG,
    "created_at":    datetime.now().isoformat(timespec="seconds"),
    "config": {
        "K_support":       K_SUPPORT,
        "Q_query":         Q_QUERY,
        "alpha_base":      ALPHA_BASE,
        "num_inner_steps": NUM_INNER_STEPS,
        "outer_lr":        OUTER_LR,
        "batch_size":      BATCH_SIZE,
        "n_iterations":    N_ITERATIONS,
        "global_seed":     GLOBAL_SEED,
        "gru_cfg":         GRU_CFG,
    },
    "test_metrics": {
        "hr1":    test_hr1,
        "hr5":    test_hr5,
        "hr10":   test_hr10,
        "ndcg10": test_ndcg10,
        "mrr":    test_mrr,
        "n":      int(len(test_labels_all)),
    },
    "val_best": {
        "best_iter":     best_iter,
        "best_val_hr10": best_val_hr10,
    },
    "ablation_support_sizes": {
        str(k): v for k, v in ablation_support_results.items()
    },
    "ablation_steps": {
        str(k): v for k, v in ablation_steps_results.items()
    },
    "checkpoint": str(CHECKPOINT_PATH),
}

write_json_atomic(RESULTS_PATH, results)
print(f"[CELL 07-13] Results saved: {RESULTS_PATH}")

# Update report.json
report = read_json(REPORT_PATH)
report["metrics"] = results["test_metrics"]
report["key_findings"] = [
    f"Standard FOMAML benchmark: HR@10={test_hr10*100:.2f}%, NDCG@10={test_ndcg10*100:.2f}%",
    f"Best val HR@10={best_val_hr10*100:.2f}% at iteration {best_iter}",
    f"Hyperparams: alpha={ALPHA_BASE}, inner_steps={NUM_INNER_STEPS}, outer_lr={OUTER_LR}, batch={BATCH_SIZE}, iters={N_ITERATIONS}",
    f"Test episodes evaluated: {len(test_labels_all)}, skipped: {n_skipped}",
]
write_json_atomic(REPORT_PATH, report)

# Update manifest.json
manifest = read_json(MANIFEST_PATH)
manifest["artifacts"].extend([
    {"type": "results_json",  "path": str(RESULTS_PATH)},
    {"type": "checkpoint",    "path": str(CHECKPOINT_PATH)},
    {"type": "report",        "path": str(REPORT_PATH)},
    {"type": "config",        "path": str(CONFIG_PATH)},
])
write_json_atomic(MANIFEST_PATH, manifest)

cell_end("CELL 07-13", t0)


[CELL 07-13] Save results JSON
[CELL 07-13] start=2026-04-09T14:12:45
[CELL 07-13] Results saved: /home/user/jamalla/anonymous-users-mooc-session-meta/reports/07_standard_maml_xuetangx/20260409_131647/results_07_standard_maml_xuetangx.json
[CELL 07-13] elapsed=0.06s  done


In [21]:
# [CELL 07-14] Update PROJECT_STATE.md with results

t0 = cell_start("CELL 07-14", "Update PROJECT_STATE.md")

state_path = PATHS["PROJECT_STATE"]

result_block = (
    f"\n\n## {NOTEBOOK_NAME} — Results\n"
    f"Run: {RUN_TAG}  |  Seed: {GLOBAL_SEED}\n"
    f"Protocol: K={K_SUPPORT} support, Q={Q_QUERY} query | Test episodes: {len(episodes_test)}\n"
    f"\n"
    f"| Metric   | Value    |\n"
    f"|----------|----------|\n"
    f"| HR@1     | {test_hr1*100:.2f}%  |\n"
    f"| HR@5     | {test_hr5*100:.2f}%  |\n"
    f"| HR@10    | {test_hr10*100:.2f}%  |\n"
    f"| NDCG@10  | {test_ndcg10*100:.2f}%  |\n"
    f"| MRR      | {test_mrr*100:.2f}%  |\n"
    f"\nBest val HR@10: {best_val_hr10*100:.2f}% @ iter {best_iter}\n"
    f"Checkpoint: {CHECKPOINT_PATH}\n"
)

if state_path.exists():
    existing = state_path.read_text(encoding="utf-8")
    marker = f"## {NOTEBOOK_NAME}"
    if marker in existing:
        # Replace existing block
        lines = existing.split("\n")
        new_lines = []
        skip = False
        for line in lines:
            if line.startswith(marker):
                skip = True
            elif skip and line.startswith("## ") and not line.startswith(marker):
                skip = False
            if not skip:
                new_lines.append(line)
        updated = "\n".join(new_lines) + result_block
    else:
        updated = existing + result_block
    state_path.write_text(updated, encoding="utf-8")
    print(f"[CELL 07-14] PROJECT_STATE.md updated: {state_path}")
else:
    state_path.write_text(result_block, encoding="utf-8")
    print(f"[CELL 07-14] PROJECT_STATE.md created: {state_path}")

cell_end("CELL 07-14", t0)


[CELL 07-14] Update PROJECT_STATE.md
[CELL 07-14] start=2026-04-09T14:12:52
[CELL 07-14] PROJECT_STATE.md updated: /home/user/jamalla/anonymous-users-mooc-session-meta/PROJECT_STATE.md
[CELL 07-14] elapsed=0.03s  done


## Notebook 07 Complete — Standard MAML (XuetangX)

**What was done:**
- Implemented First-Order MAML (FOMAML) with `create_graph=False`
- `functional_forward` uses `torch.func.functional_call` for native CUDA GRU speed
- Inner loop: 5 gradient steps on support set with fixed `alpha_base=0.01`
- Outer loop: query loss accumulated across batch, single meta-optimizer step
- Validation every 100 iterations; best HR@10 checkpoint saved at iter **2800**
- Ablations: K ∈ {1,3,5,10} support sizes and steps ∈ {1,3,5,10}

**Locked hyperparameters:**
```
alpha_base=0.01  num_inner_steps=5  outer_lr=0.001
batch_size=32    n_iterations=3000  seed=20260107
```

**Test Results (K=5 support, Q=10 query, 986 episodes):**
| Metric | Value |
|--------|-------|
| HR@1 | 23.36% |
| HR@5 | 39.54% |
| **HR@10** | **47.46%** |
| **NDCG@10** | **34.35%** |
| MRR | 31.49% |

**Next:** `08_warmstart_maml_xuetangx.ipynb` (ablation: warm-start init without SRS)
